# MAS v2 Baseline — 3DSRBench

Main architecture (no Trust Score): Head Agent → 3 Specialists → Final Reasoning Agent.

**Benchmark:** 3DSRBench (ccvl/3DSRBench)  
**Testing only** — no train/test split  
**Sample sizes:** 10, 50, 100

In [ ]:
import sys
from pathlib import Path

# Project root
ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "run_eval_mas_v2.py").exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from run_eval_mas_v2 import build_runners, run_test_only

In [ ]:
# H100: use local DeepSeek-R1 (no API server)
head_gen, spec_gen, reason_gen = build_runners(
    specialist_device="cuda",
    use_local_reasoning=True,
    reasoning_local_model_id="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
)

In [ ]:
BENCHMARK = "3dsrbench"
SAMPLE_SIZES = [10, 50, 100]
SEED = 42
OUTPUT_BASE = ROOT / "results" / "mas_v2_baseline" / BENCHMARK

In [ ]:
results_summary = []

for n in SAMPLE_SIZES:
    print(f"\n{'='*60}")
    print(f"3DSRBench | {n} samples")
    print("="*60)
    out_dir = str(OUTPUT_BASE / f"{n}samples")
    out = run_test_only(
        benchmark=BENCHMARK,
        head_generate=head_gen,
        specialist_generate=spec_gen,
        reasoning_generate=reason_gen,
        max_samples=n,
        seed=SEED,
        output_dir=out_dir,
    )
    m = out["metrics"]
    results_summary.append({
        "samples": n,
        "accuracy": m["accuracy"],
        "correct": m["correct"],
        "total": m["total"],
    })

In [ ]:
import pandas as pd

df = pd.DataFrame(results_summary)
df["accuracy%"] = (df["accuracy"] * 100).round(1)
display(df[["samples", "total", "correct", "accuracy%"]])